# Introduction to Regression Algorithms

**What this notebook covers:**
- What regression is and how it differs from classification
- Implementing Linear Regression from scratch using the Normal Equation
- Comparing multiple regression algorithms using scikit-learn
- Evaluating models using MSE, RMSE, MAE, and R²
- Visualizing predictions and hyperparameter effects

**Prerequisites:**
- Introduction to Supervised Learning
- Python, NumPy, Pandas basics
- Basic statistics (mean, variance, correlation)
- Basic linear algebra (matrix multiplication)

---

**Dataset:** House Prices — Advanced Regression Techniques  
🔗 [Kaggle Dataset Link](https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data)  
*Credits: Kaggle / Dean De Cock — Ames Housing dataset with 79 features describing residential homes in Ames, Iowa.*

In [ ]:
# --- All Imports ---
import numpy as np                            # Numerical operations and matrix math
import pandas as pd                           # Data loading and manipulation
import matplotlib.pyplot as plt               # Plotting and visualization
import seaborn as sns                         # Statistical visualizations

from sklearn.linear_model import LinearRegression, Ridge, Lasso   # Regression models
from sklearn.tree import DecisionTreeRegressor                     # Tree-based regression
from sklearn.model_selection import train_test_split, cross_val_score  # Splitting and CV
from sklearn.preprocessing import StandardScaler                   # Feature scaling
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score  # Metrics

np.random.seed(42)  # Reproducibility for all random operations

## Part 1: Theory Recap

Five key things to remember about regression algorithms:

- **Regression = predicting a continuous number**: Unlike classification (categories), regression outputs quantities like price, temperature, or score.
- **All regression algorithms minimize a loss function**: Most commonly MSE = (1/n)×Σ(y - ŷ)² — the average squared difference between predicted and actual values.
- **R² score measures explanatory power**: R²=1 is perfect; R²=0 means the model is no better than always predicting the mean; negative R² means the model is actively harmful.
- **Regularization controls overfitting**: Ridge (L2) shrinks all coefficients; Lasso (L1) forces some to exactly zero (feature selection); Elastic Net combines both.
- **Always beat the baseline**: A good regression model must outperform simply predicting the mean of y for every sample — that is your minimum benchmark.

## Loading the Dataset

We use the **Ames Housing dataset** — a rich real-world regression dataset with 79 features.
- **Input features (X):** House characteristics — size, quality, location, rooms, year built, etc.
- **Target variable (y):** `SalePrice` — the actual sale price of each house in USD

This is a regression problem: given house characteristics, predict its sale price.

In [ ]:
# Load the Ames Housing dataset from a local sourc
df = pd.read_csv("train.csv")

print("Shape:", df.shape)
print("\n--- First 5 Rows ---")
display(df.head())

print("\n--- Dataset Info ---")
df.info()

print("\n--- Statistical Summary (Numerical Columns) ---")
display(df.describe())

print("\n--- Target Variable: SalePrice ---")
print(f"Mean Sale Price : ${df['SalePrice'].mean():,.0f}")
print(f"Min  Sale Price : ${df['SalePrice'].min():,.0f}")
print(f"Max  Sale Price : ${df['SalePrice'].max():,.0f}")
print(f"Std  Sale Price : ${df['SalePrice'].std():,.0f}")

## Preprocessing

The Ames Housing dataset has 79 features with mixed types and missing values. We need to:
1. **Select key numerical features** — focus on the most predictive ones for clarity
2. **Handle missing values** — fill with median (robust to outliers)
3. **Encode categorical variables** — convert text categories to numbers
4. **Scale features** — normalize so all features contribute equally to the model
5. **Split into train/test** — honest evaluation on unseen data

In [ ]:
# Step 1: Select the most predictive numerical features for this introduction
# (keeping it focused — full feature engineering is a separate topic)
features = [
    'Gr Liv Area',      # Above ground living area (sq ft) — strongest predictor
    'Overall Qual',     # Overall material and finish quality (1-10 scale)
    'Year Built',       # Original construction year
    'Total Bsmt SF',    # Total basement area (sq ft)
    'Garage Cars',      # Garage capacity in cars
    'Full Bath',        # Number of full bathrooms
    'TotRms AbvGrd',    # Total rooms above ground
    'Fireplaces'        # Number of fireplaces
]
target = 'SalePrice'

# Step 2: Subset to selected features and target
df_clean = df[features + [target]].copy()

# Step 3: Fill missing values with column median (robust to outliers)
for col in features:
    df_clean[col].fillna(df_clean[col].median(), inplace=True)

# Step 4: Separate features and target
X = df_clean[features].values
y = df_clean[target].values

# Step 5: Train/test split — 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 6: Scale features to mean=0, std=1
# IMPORTANT: Fit scaler ONLY on training data to prevent data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)  # Only transform, never fit on test

print("Training set:", X_train_scaled.shape)
print("Test set    :", X_test_scaled.shape)
print("Missing values remaining:", df_clean.isnull().sum().sum())
print(f"\nBaseline (predict mean always): RMSE = ${np.std(y_test):,.0f}")

## Part 2: From Scratch Implementation

We implement **Linear Regression from scratch** using the **Normal Equation** — the closed-form mathematical solution.

The Normal Equation directly computes the optimal coefficients θ that minimize MSE:

```
θ = (XᵀX)⁻¹ Xᵀy
```

This is exactly what `sklearn.LinearRegression().fit()` computes internally. Building it from scratch shows you there is no magic — just matrix algebra.

In [ ]:
class LinearRegressionFromScratch:
    """
    Linear Regression implemented from scratch using the Normal Equation.
    θ = (XᵀX)⁻¹ Xᵀy — computes optimal coefficients analytically.
    """

    def __init__(self):
        self.theta = None   # Will hold [intercept, coef1, coef2, ...]

    def fit(self, X, y):
        # Step 1: Add bias column (column of 1s) for the intercept term
        # INTERVIEW NOTE: Without this, the regression line is forced through the origin
        n_samples = X.shape[0]
        X_b = np.c_[np.ones((n_samples, 1)), X]   # X with bias: shape (n, p+1)

        # Step 2: Apply the Normal Equation: θ = (XᵀX)⁻¹ Xᵀy
        # np.linalg.inv computes matrix inverse
        # INTERVIEW NOTE: This is O(p³) — expensive for very high-dimensional data.
        # For large p, use gradient descent instead.
        self.theta = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y

    def predict(self, X):
        # Add bias column and compute dot product with learned coefficients
        n_samples = X.shape[0]
        X_b = np.c_[np.ones((n_samples, 1)), X]
        return X_b @ self.theta   # ŷ = Xθ

    def score(self, X, y):
        # Compute R² score manually
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)          # Residual sum of squares
        ss_tot = np.sum((y - np.mean(y)) ** 2)      # Total sum of squares
        return 1 - (ss_res / ss_tot)                 # R² = 1 - SS_res/SS_tot

# Train the from-scratch model
lr_scratch = LinearRegressionFromScratch()
lr_scratch.fit(X_train_scaled, y_train)

print("Linear Regression from scratch trained!")
print(f"Intercept : {lr_scratch.theta[0]:,.2f}")
print("\nFeature Coefficients:")
for feat, coef in zip(features, lr_scratch.theta[1:]):
    print(f"  {feat:20s}: {coef:>10,.2f}")

In [ ]:
# Evaluate the from-scratch model
y_pred_scratch = lr_scratch.predict(X_test_scaled)

mse_scratch  = mean_squared_error(y_test, y_pred_scratch)
rmse_scratch = np.sqrt(mse_scratch)
mae_scratch  = mean_absolute_error(y_test, y_pred_scratch)
r2_scratch   = r2_score(y_test, y_pred_scratch)

print("=== From Scratch Linear Regression Results ===")
print(f"MSE  : {mse_scratch:>15,.2f}")
print(f"RMSE : ${rmse_scratch:>14,.2f}  ← average prediction error in dollars")
print(f"MAE  : ${mae_scratch:>14,.2f}  ← median prediction error in dollars")
print(f"R²   : {r2_scratch:>15.4f}  ← model explains {r2_scratch*100:.1f}% of price variance")

## Part 3: Sklearn Implementation

Scikit-learn provides optimized implementations of multiple regression algorithms under a unified API. Here we compare four algorithms on the same dataset:

- **Linear Regression** — our baseline, no regularization
- **Ridge Regression** — adds L2 penalty, shrinks all coefficients
- **Lasso Regression** — adds L1 penalty, forces some coefficients to zero
- **Decision Tree Regressor** — non-linear, no assumptions about relationships

Comparing these shows how algorithm choice impacts performance on the same data.

In [ ]:
# Define all models to compare
models = {
    'Linear Regression'      : LinearRegression(),
    'Ridge (α=1.0)'          : Ridge(alpha=1.0),
    'Lasso (α=100)'          : Lasso(alpha=100),
    'Decision Tree (depth=5)': DecisionTreeRegressor(max_depth=5, random_state=42)
}

results = []

for name, model in models.items():
    # Train each model
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    # Compute all metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    cv   = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2').mean()

    results.append({'Model': name, 'RMSE ($)': f"{rmse:,.0f}", 'MAE ($)': f"{mae:,.0f}",
                    'R²': f"{r2:.4f}", 'CV R² (5-fold)': f"{cv:.4f}"})

# Add from-scratch result for comparison
results.append({'Model': 'Linear Reg (Scratch)', 'RMSE ($)': f"{rmse_scratch:,.0f}",
                'MAE ($)': f"{mae_scratch:,.0f}", 'R²': f"{r2_scratch:.4f}", 'CV R² (5-fold)': 'N/A'})

results_df = pd.DataFrame(results)
print("=== Algorithm Comparison ===")
display(results_df)
print("\nNote: Sklearn LinearRegression and Scratch should give identical R² — verifying our implementation!")

In [ ]:
# --- Visualisation 1: Actual vs Predicted Prices ---
# A perfect model would have all points on the diagonal line
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Get predictions from Linear Regression and Decision Tree for comparison
lr_sklearn = models['Linear Regression']
dt_sklearn = models['Decision Tree (depth=5)']

for ax, model, title in zip(axes,
                             [lr_sklearn, dt_sklearn],
                             ['Linear Regression', 'Decision Tree Regressor']):
    y_pred = model.predict(X_test_scaled)
    r2 = r2_score(y_test, y_pred)

    ax.scatter(y_test, y_pred, alpha=0.5, color='steelblue', edgecolors='white', linewidth=0.5)
    # Perfect prediction line
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
    ax.set_title(f'{title}\nR² = {r2:.4f}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Actual Sale Price ($)', fontsize=11)
    ax.set_ylabel('Predicted Sale Price ($)', fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('Actual vs Predicted House Prices — Regression Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# --- Visualisation 2: Algorithm RMSE Comparison ---
# Bar chart to visually compare all algorithm errors
algo_names = ['Linear\nRegression', 'Ridge\n(α=1.0)', 'Lasso\n(α=100)', 'Decision Tree\n(depth=5)']
rmse_values = []
for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    rmse_values.append(np.sqrt(mean_squared_error(y_test, y_pred)))

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['steelblue', 'salmon', 'green', 'purple']
bars = ax.bar(algo_names, rmse_values, color=colors, edgecolor='black', width=0.5)
ax.set_title('RMSE Comparison Across Regression Algorithms', fontsize=13, fontweight='bold')
ax.set_xlabel('Algorithm', fontsize=11)
ax.set_ylabel('RMSE ($ error)', fontsize=11)
for bar, val in zip(bars, rmse_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f'${val:,.0f}', ha='center', fontsize=10, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Part 4: Hyperparameter Experiments

The most important hyperparameter in regularized regression is **α (alpha)** — the regularization strength.

- **α = 0:** No regularization → same as plain Linear Regression
- **Small α:** Mild regularization → slightly shrunk coefficients
- **Large α:** Strong regularization → heavily shrunk coefficients → underfitting risk

We experiment with Ridge and Lasso across a wide range of α values to find the sweet spot.

In [ ]:
# Experiment: Vary alpha for Ridge and Lasso — observe effect on R²
alphas = np.logspace(-2, 5, 50)   # 0.01 to 100,000 on log scale

ridge_r2_train, ridge_r2_cv = [], []
lasso_r2_train, lasso_r2_cv = [], []

for alpha in alphas:
    # Ridge
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)
    ridge_r2_train.append(r2_score(y_train, ridge.predict(X_train_scaled)))
    ridge_r2_cv.append(cross_val_score(ridge, X_train_scaled, y_train, cv=5, scoring='r2').mean())

    # Lasso
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train_scaled, y_train)
    lasso_r2_train.append(r2_score(y_train, lasso.predict(X_train_scaled)))
    lasso_r2_cv.append(cross_val_score(lasso, X_train_scaled, y_train, cv=5, scoring='r2').mean())

# Plot results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ridge plot
axes[0].semilogx(alphas, ridge_r2_train, label='Train R²', color='blue', linewidth=2)
axes[0].semilogx(alphas, ridge_r2_cv, label='CV R² (5-fold)', color='orange', linewidth=2)
best_ridge_alpha = alphas[np.argmax(ridge_r2_cv)]
axes[0].axvline(x=best_ridge_alpha, color='red', linestyle='--', label=f'Best α = {best_ridge_alpha:.2f}')
axes[0].set_title('Ridge Regression: α vs R²', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Alpha (regularization strength) — log scale', fontsize=11)
axes[0].set_ylabel('R² Score', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Lasso plot
axes[1].semilogx(alphas, lasso_r2_train, label='Train R²', color='green', linewidth=2)
axes[1].semilogx(alphas, lasso_r2_cv, label='CV R² (5-fold)', color='purple', linewidth=2)
best_lasso_alpha = alphas[np.argmax(lasso_r2_cv)]
axes[1].axvline(x=best_lasso_alpha, color='red', linestyle='--', label=f'Best α = {best_lasso_alpha:.2f}')
axes[1].set_title('Lasso Regression: α vs R²', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Alpha (regularization strength) — log scale', fontsize=11)
axes[1].set_ylabel('R² Score', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Hyperparameter Tuning: Effect of Regularization Strength (α)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Best Ridge α: {best_ridge_alpha:.4f} → CV R²: {max(ridge_r2_cv):.4f}")
print(f"Best Lasso α: {best_lasso_alpha:.4f} → CV R²: {max(lasso_r2_cv):.4f}")

## Part 5: Interview Corner

**Question: What is R² score and what does it actually mean? Can it be negative?**

This catches many candidates off guard. Here is a complete FAANG-level answer:

**Definition:**  
R² (coefficient of determination) measures the proportion of variance in the target variable y that is explained by the model.

```
R² = 1 - (SS_residual / SS_total)
   = 1 - Σ(y - ŷ)² / Σ(y - ȳ)²
```

**Intuition:**
- SS_total = total variance in y (how spread out the actual values are)
- SS_residual = variance the model *failed* to explain (prediction errors)
- R² = fraction of variance the model *did* explain

**Can R² be negative? YES.**  
If SS_residual > SS_total, R² becomes negative. This means the model is worse than just predicting the mean ȳ for every sample. It can happen when:
- You evaluate on a very different distribution than training data
- The model drastically overfits and extrapolates poorly
- Wrong features are used entirely

**Interview tip:** Always report R² alongside RMSE. A model can have high R² but still have RMSE of $50,000 — which might be unacceptable in practice depending on the context.

## Key Takeaways

Remember these 5 bullets for placement interviews:

- **Regression predicts continuous values**: Output is a number (price, score, demand) — not a category. Loss function is MSE; evaluation uses RMSE, MAE, and R².
- **Linear Regression uses the Normal Equation θ = (XᵀX)⁻¹Xᵀy**: This is the closed-form optimal solution that sklearn's `fit()` computes internally — no iteration needed.
- **Regularization prevents overfitting**: Ridge (L2) shrinks all coefficients; Lasso (L1) forces some to zero (automatic feature selection); Elastic Net combines both. α controls strength.
- **Always compare against the baseline**: A regression model must beat predicting the mean of y every time. If it doesn't, the model adds no value.
- **R² can be negative**: If the model is worse than predicting the mean, R² goes below zero. High R² on training data with low R² on test data signals overfitting.